# VLM Bias Prediction - Hugging Face Hosted Inference

This notebook keeps your existing dataset loading and image loop flow, and swaps only model init/inference to Hugging Face token-based hosted inference.

In [1]:
# Install minimal dependencies for hosted inference
!pip install -q pandas pillow huggingface_hub

In [2]:
import base64
import json
import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from huggingface_hub import InferenceClient

# Keep your prompt and parsing logic
PROMPT   = """Look at this image carefully.

How many legs does this animal have?
Count ALL visible legs.

Respond with only a number."""
# Set your token (recommended: set as environment variable in Colab/local runtime)
# os.environ["HF_TOKEN"] = "hf_xxx"
HF_TOKEN = os.environ.get("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Set os.environ['HF_TOKEN'] before running.")

# Keep dataset loading structure; choose local path or Colab/Drive path
DATA_BASE = "data/raw"  # change if needed for Colab path
IMAGE_DIR = os.path.join(DATA_BASE, "counterfactual_images")
RESULTS_DIR = os.path.join(DATA_BASE, "results_hf")
os.makedirs(RESULTS_DIR, exist_ok=True)


def parse_legs(text: str):
    if not text:
        return None

    raw = text.strip().lower()

    # 1) Best signal for your current Gemma style output:
    #    count how many JSON labels mention "leg".
    label_hits = re.findall(r'"label"\s*:\s*"[^"]*leg[^"]*"', raw)
    if label_hits:
        return len(label_hits)

    # 2) Look for explicit final-count style phrases.
    phrase_patterns = [
        r"(?:total\s*(?:legs?)?\s*(?:is|:)?\s*)(\d{1,2})\b",
        r"(?:there\s+(?:are|is)\s*)(\d{1,2})\s+legs?\b",
        r"(?:answer\s*(?:is|:)?\s*)(\d{1,2})\b",
        r"(?:count\s*(?:is|:)?\s*)(\d{1,2})\b",
    ]
    for pat in phrase_patterns:
        m = re.search(pat, raw)
        if m:
            return int(m.group(1))

    # 3) Word-number fallback for final statements.
    word_to_int = {
        "zero": 0,
        "one": 1,
        "two": 2,
        "three": 3,
        "four": 4,
        "five": 5,
        "six": 6,
        "seven": 7,
        "eight": 8,
        "nine": 9,
        "ten": 10,
    }
    for word, value in word_to_int.items():
        if re.search(rf"\b{word}\s+legs?\b", raw):
            return value

    # 4) Last-resort numeric parse: remove JSON coordinate blocks and bullet markers,
    #    then take the last small integer token (avoid grabbing list item '1.' first).
    cleaned = re.sub(r"\{[^{}]*\}", " ", raw)
    cleaned = re.sub(r"(?m)^\s*\d+\.\s+", " ", cleaned)
    nums = [int(n) for n in re.findall(r"\b\d{1,2}\b", cleaned)]
    if nums:
        return nums[-1]

    return None


def parse_filename(fname: str):
    name = Path(fname).stem
    m = re.match(r"^(.+?)_(\d+)legs_(natural|plain)$", name)
    if m:
        return m.group(1), int(m.group(2)), m.group(3)
    return name, None, None


def get_all_images():
    return sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))])


def save_results(records, csv_path):
    pd.DataFrame(records).to_csv(csv_path, index=False)


def append_jsonl_log(log_path: str, payload: dict):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=True) + "\n")


print(f"Images found: {len(get_all_images())}")
print(f"Results directory: {RESULTS_DIR}")

c:\Users\Imsac\Desktop\Project_X\X5\vlm_bias_prediction\.peVLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Images found: 60
Results directory: data/raw\results_hf


In [3]:
# --- New hosted inference block (replaces local model init + model.generate) ---

def infer_legs_hf(image_path: str, model_id: str, prompt: str, hf_token: str, provider: str = "together"):
    client = InferenceClient(provider=provider, api_key=hf_token)

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    ext = Path(image_path).suffix.lower()
    mime = "image/png" if ext == ".png" else "image/jpeg"
    image_data_url = f"data:{mime};base64,{base64.b64encode(image_bytes).decode('utf-8')}"

    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_data_url}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
        max_tokens=128,
    )

    message = response.choices[0].message
    message_content = message.content

    # Provider responses can be string, list of chunks, or None.
    content_text = ""
    if isinstance(message_content, str):
        content_text = message_content.strip()
    elif isinstance(message_content, list):
        parts = []
        for item in message_content:
            if isinstance(item, dict):
                text_part = item.get("text")
                if text_part:
                    parts.append(str(text_part))
            elif item:
                parts.append(str(item))
        content_text = " ".join(parts).strip()
    elif message_content is not None:
        content_text = str(message_content).strip()

    # Some Together-hosted models place text in `reasoning` while content is empty.
    if content_text:
        return content_text

    reasoning_text = getattr(message, "reasoning", None)
    if reasoning_text:
        return str(reasoning_text).strip()

    return ""

In [5]:
# Evaluate one or more HF models using the same image loop
MODELS = [
    # Already run:
    # {"name": "Gemma-4-31B", "hf_id": "google/gemma-4-31B-it", "provider": "together"},
    # {"name": "Qwen3.5-9B", "hf_id": "Qwen/Qwen3.5-9B", "provider": "together"},
    # {"name": "Kimi-K2.6", "hf_id": "moonshotai/Kimi-K2.6", "provider": "together"},

    # Run now:
    {"name": "Qwen3-VL-235B-A22B", "hf_id": "Qwen/Qwen3-VL-235B-A22B-Instruct", "provider": "novita"},
    {"name": "Llama-4-Scout", "hf_id": "meta-llama/Llama-4-Scout-17B-16E-Instruct", "provider": "groq"},
]

for m in MODELS:
    model_name = m["name"]
    model_id = m["hf_id"]
    provider = m.get("provider", "together")
    model_slug = model_name.lower().replace('-', '_').replace('.', '')
    csv_out = os.path.join(RESULTS_DIR, f"{model_slug}_results.csv")
    log_out = os.path.join(RESULTS_DIR, f"{model_slug}_outputs.jsonl")

    records = list(pd.read_csv(csv_out).to_dict("records")) if os.path.exists(csv_out) else []
    done = {r["image"] for r in records}

    print(f"\nRunning {model_name} ({model_id}) via provider={provider}")
    print(f"Output log: {log_out}")
    for fname in get_all_images():
        if fname in done:
            print(f"  skip {fname}")
            continue

        image_path = os.path.join(IMAGE_DIR, fname)
        concept, true_legs, bg = parse_filename(fname)

        error_msg = None
        try:
            text = infer_legs_hf(image_path, model_id, PROMPT, HF_TOKEN, provider=provider)
            pred = parse_legs(text)
        except Exception as e:
            text = ""
            pred = None
            error_msg = str(e)

        if error_msg is not None:
            status = "api_error"
        elif not str(text).strip():
            status = "empty_output"
        elif pred is None:
            status = "parse_failed"
        else:
            status = "ok"

        is_biased = int(pred != true_legs) if pred is not None else 1

        records.append(
            {
                "image": fname,
                "concept": concept,
                "true_legs": true_legs,
                "background": bg,
                "predicted": pred,
                "raw_output": text,
                "status": status,
                "is_biased": is_biased,
                "model": model_name,
                "hf_id": model_id,
                "provider": provider,
                "error": error_msg,
            }
        )

        append_jsonl_log(
            log_out,
            {
                "ts": datetime.utcnow().isoformat() + "Z",
                "image": fname,
                "concept": concept,
                "true_legs": true_legs,
                "background": bg,
                "predicted": pred,
                "raw_output": text,
                "status": status,
                "error": error_msg,
                "is_biased": is_biased,
                "model": model_name,
                "hf_id": model_id,
                "provider": provider,
                "prompt": PROMPT,
                "quantization": "none-hosted-inference",
            },
        )

        save_results(records, csv_out)
        if pred is None:
            print(f"{model_name} | {fname} | status={status} | pred=None | true={true_legs} | bias={is_biased} | error={error_msg} | raw={text}")
        else:
            print(f"{model_name} | {fname} | status={status} | pred={pred} | true={true_legs} | bias={is_biased}")

    print(f"Done: {csv_out}")


Running Qwen3-VL-235B-A22B (Qwen/Qwen3-VL-235B-A22B-Instruct) via provider=novita
Output log: data/raw\results_hf\qwen3_vl_235b_a22b_outputs.jsonl


C:\Users\Imsac\AppData\Local\Temp\ipykernel_15140\1822172467.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat() + "Z",


Qwen3-VL-235B-A22B | binturong_3legs_natural.png | status=ok | pred=4 | true=3 | bias=1


C:\Users\Imsac\AppData\Local\Temp\ipykernel_15140\1822172467.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat() + "Z",


Qwen3-VL-235B-A22B | binturong_3legs_plain.png | status=ok | pred=4 | true=3 | bias=1


...

C:\Users\Imsac\AppData\Local\Temp\ipykernel_15140\1822172467.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat() + "Z",


Llama-4-Scout | serval_3legs_plain.png | status=ok | pred=3 | true=3 | bias=0


C:\Users\Imsac\AppData\Local\Temp\ipykernel_15140\1822172467.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat() + "Z",


Llama-4-Scout | serval_5legs_natural.jpg | status=ok | pred=3 | true=5 | bias=1
Llama-4-Scout | serval_5legs_plain.png | status=ok | pred=3 | true=5 | bias=1
Done: data/raw\results_hf\llama_4_scout_results.csv


C:\Users\Imsac\AppData\Local\Temp\ipykernel_15140\1822172467.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat() + "Z",


## Notes

- This notebook intentionally removes local `from_pretrained(...)` model loading.
- Inference happens through Hugging Face hosted API using `HF_TOKEN`.
- Per-image raw outputs are logged to `*_outputs.jsonl` files in `results_hf`.
- This hosted path does not use local quantization in this notebook.